In [1]:
import nltk
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

/home/web-h-006/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/web-h-006/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
class SemanticChunker:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", threshold=0.75):
        self.threshold = threshold
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
            input_mask_expanded.sum(1), min=1e-9
        )

    def embed_sentences(self, sentences):
        encoded_input = self.tokenizer(
            sentences,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        sentence_embeddings = self._mean_pooling(
            model_output,
            encoded_input["attention_mask"]
        )

        return sentence_embeddings.cpu().numpy()

    def chunk(self, text):
        sentences = nltk.sent_tokenize(text)
        
        if len(sentences) <= 1:
            return [text]

        embeddings = self.embed_sentences(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity(
                [embeddings[i - 1]],
                [embeddings[i]]
            )[0][0]

            if sim < self.threshold:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
            else:
                current_chunk.append(sentences[i])

        chunks.append(" ".join(current_chunk))
        return chunks

In [12]:
text = """
Machine learning models rely on large datasets. 
They improve performance through gradient descent. 
Bananas are rich in potassium and grow in tropical climates. 
Fruit consumption improves overall health.
"""

chunker = SemanticChunker(threshold=0.70)
chunks = chunker.chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


Chunk 1:

Machine learning models rely on large datasets.

Chunk 2:
They improve performance through gradient descent.

Chunk 3:
Bananas are rich in potassium and grow in tropical climates.

Chunk 4:
Fruit consumption improves overall health.


In [16]:
text = """
Artificial intelligence systems have evolved from rule-based expert systems into large-scale neural architectures capable of generating language, images, and even software code. Modern transformer models rely on attention mechanisms that dynamically weigh contextual relationships between tokens, allowing them to model long-range dependencies across entire documents. Training these systems requires enormous datasets and computational infrastructure, often distributed across data centers containing thousands of GPUs operating in parallel. Optimization techniques such as gradient descent, learning rate scheduling, and regularization play a central role in shaping how these models converge toward useful representations. As models scale, emergent behaviors sometimes appear, raising questions about interpretability, alignment, and robustness.

Meanwhile, the human brain operates on entirely different biological principles, using electrochemical signals transmitted across networks of neurons. Synaptic plasticity enables learning by strengthening or weakening connections based on experience, a mechanism often summarized by the phrase “neurons that fire together wire together.” Unlike artificial neural networks, biological systems consume remarkably little energy while performing complex perceptual and cognitive tasks. Neuroscientists continue to debate how consciousness arises from neural activity, and whether subjective experience can ever be fully explained through physical processes alone.

Centuries before digital computation, ancient civilizations engineered monumental structures using surprisingly sophisticated mathematics and astronomy. The construction of pyramids required precise alignment with celestial bodies, careful logistical planning, and coordinated labor forces. Trade networks connected distant regions, enabling the exchange of metals, spices, and cultural ideas. Written language systems evolved to record laws, religious texts, and commercial transactions, preserving knowledge across generations. Over time, empires rose and fell, shaped by geography, resource distribution, and political organization.

In the modern era, industrialization dramatically increased humanity’s capacity to extract and consume natural resources. Fossil fuel combustion released vast quantities of carbon dioxide into the atmosphere, altering global climate systems. Rising temperatures contribute to melting glaciers, shifting weather patterns, and biodiversity loss. Scientists model these changes using complex simulations that integrate atmospheric chemistry, ocean currents, and ecological feedback loops. Policymakers face difficult trade-offs between economic development and environmental sustainability.

Philosophers, observing these transformations, ask enduring questions about responsibility, progress, and the meaning of intelligence. If machines can simulate reasoning, does that diminish the uniqueness of human thought? Ethical frameworks such as utilitarianism and deontology offer competing approaches to evaluating technological consequences. Some argue that innovation inevitably disrupts established norms, while others contend that moral reflection must guide scientific advancement. The tension between capability and wisdom remains unresolved.

Beyond Earth, space agencies and private companies design missions to explore Mars and the outer planets. Robotic probes analyze soil composition, atmospheric chemistry, and radiation exposure. Engineers experiment with reusable rockets to reduce launch costs and enable sustained exploration. The search for extraterrestrial life continues through radio telescopes and planetary missions, driven by the possibility that biology may not be unique to Earth. Humanity’s curiosity pushes outward, even as it grapples with challenges at home.
"""

chunker = SemanticChunker(threshold=0.10)
chunks = chunker.chunk(text)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


Chunk 1:

Artificial intelligence systems have evolved from rule-based expert systems into large-scale neural architectures capable of generating language, images, and even software code. Modern transformer models rely on attention mechanisms that dynamically weigh contextual relationships between tokens, allowing them to model long-range dependencies across entire documents. Training these systems requires enormous datasets and computational infrastructure, often distributed across data centers containing thousands of GPUs operating in parallel. Optimization techniques such as gradient descent, learning rate scheduling, and regularization play a central role in shaping how these models converge toward useful representations. As models scale, emergent behaviors sometimes appear, raising questions about interpretability, alignment, and robustness. Meanwhile, the human brain operates on entirely different biological principles, using electrochemical signals transmitted across networks o

1. Consecutive Sentence Similarity Method
2. Windowed Similarity Drop
3. Hierarchical Agglomerative Chunking

In [19]:
from langchain_experimental.text_splitter import SemanticChunker